# Nova Papers — P1 web demo on Colab

Deploys the P1 GRPO checkpoint behind the repo's own web UI on a free Colab T4, and prints a temporary public URL.
See `docs/COLAB_DEMO.md` for the full walkthrough and troubleshooting.

**Before you start:** *Runtime → Change runtime type → T4 GPU.* Then run the cells top to bottom. Step 3 needs a manual upload. First run takes about 25–30 minutes.

## 1. Clone the repo and install Python dependencies (3–5 min)
Red pip "dependency conflict" messages about Colab's preinstalled packages (gradio, cudf, google-adk, …) are harmless.

In [ ]:
%%bash
cd /content
[ -d nova-papers ] || git clone -q https://github.com/hxhuser2003/nova-papers
pip install -q vllm fastapi uvicorn httpx pydantic-settings pymupdf rank_bm25

## 2. Install Node 22 and build the frontend (2–3 min)
The frontend's Vite 8 needs Node ≥ 20.19; Colab's built-in Node is older. `apt` warnings in the output are harmless.

In [ ]:
%%bash
curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null
apt-get install -y -qq nodejs > /dev/null
node --version
cd /content/nova-papers/frontend
npm ci --silent && npm run build 2>&1 | tail -3
ls dist

## 3. Upload and unzip the P1 checkpoint
Open the 📁 Files panel and drag `p1-checkpoint.zip` into `/content` (next to `nova-papers`, not inside it). It must contain `p1-checkpoint/adapter_config.json` and `p1-checkpoint/adapter_model.safetensors`. Run the cell once the upload finishes.

In [ ]:
%%bash
cd /content && unzip -q -o p1-checkpoint.zip -d /content/ckpt
ls /content/ckpt/p1-checkpoint/

## 4. Merge P1 into the base model and start vLLM (10–15 min)
On a T4 with current vLLM, serving the adapter at runtime (`--enable-lora`) fails at engine start-up, so the adapter is merged into Qwen2.5-3B-Instruct first and served as a plain model under the name `nova-p1` (same weights, same outputs). Same approach as `scripts/merge_lora_and_push.py`.

Colab ships an old `torchao` that makes current `peft` raise an ImportError; the demo doesn't use torchao, so it is uninstalled first.

In [ ]:
%%bash
pkill -f "vllm serve" ; sleep 3
pip uninstall -y -q torchao
pip install -q peft
cd /content
[ -f /content/p1-merged/config.json ] || python - <<'PY'
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", dtype=torch.float16, device_map="cuda")
model = PeftModel.from_pretrained(base, "/content/ckpt/p1-checkpoint").merge_and_unload()
model.save_pretrained("/content/p1-merged", safe_serialization=True)
AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct").save_pretrained("/content/p1-merged")
print("merged -> /content/p1-merged")
PY
nohup vllm serve /content/p1-merged --served-model-name nova-p1 --dtype half \
  --max-model-len 8192 --gpu-memory-utilization 0.85 --port 8000 > vllm.log 2>&1 &
for i in $(seq 1 60); do
  curl -s localhost:8000/v1/models | grep -q nova-p1 && echo "OK: vLLM ready (merged P1)" && exit 0
  grep -aq "Engine core initialization failed" vllm.log && echo "FAILED" && grep -a "EngineCore" vllm.log | grep -av INFO | tail -15 && exit 1
  sleep 10
done
echo "Still loading - run the next cell in a minute"

In [ ]:
# Only needed if the cell above said "Still loading". Do not re-run the cell above: it would start a second vLLM.
!curl -s localhost:8000/v1/models | grep -q nova-p1 && echo "OK: vLLM ready" || (echo "still loading"; tail -3 /content/vllm.log)

## 5. Start the web backend (pointed at `nova-p1`)

In [ ]:
%%bash
cd /content/nova-papers
cat > .env <<'ENV'
VLLM_BASE_URL=http://localhost:8000
VLLM_MODEL=nova-p1
VLLM_TIMEOUT_SECONDS=120
ENV
nohup python -m uvicorn backend.main:app --host 0.0.0.0 --port 8127 > /content/web.log 2>&1 &
sleep 15
curl -s localhost:8127/api/health && echo && echo "OK: web backend up" || tail -20 /content/web.log

## 6. Get a public URL
Prints an `https://<random>.trycloudflare.com` link to the demo. It has **no password** — share it privately, and disconnect the runtime when done. The link dies when the runtime does.

In [ ]:
%%bash
cd /content
[ -x cloudflared ] || { wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared; }
nohup ./cloudflared tunnel --url http://localhost:8127 > tunnel.log 2>&1 &
sleep 12
grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | head -1

## Demo questions
The app picks the paper from the question, so QASPER-style questions ("What datasets are used?") can land on the wrong paper. Name the paper in the question, e.g. *In the paper "PO-EMO: Conceptualization, Annotation, and Modeling of Aesthetic Emotions", what datasets are used?* This prints three sample questions with their paper and gold answer:

In [ ]:
!python -c "
import json,random
P={json.loads(l)['paper_id']:json.loads(l)['title'] for l in open('/content/nova-papers/corpus/qasper/papers.jsonl')}
for q in random.sample([json.loads(l) for l in open('/content/nova-papers/eval/sample_questions.jsonl')],3):
    print('paper:',P.get(q['paper_id']));print('question:',q['question']);print('gold:',[a.get('answer_text') or 'Unanswerable' for a in q['answers']]);print()
"

## Logs (if something fails)

In [ ]:
!tail -30 /content/vllm.log | cut -c1-250; echo ----; tail -30 /content/web.log